In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Set legacy Keras mode BEFORE any TensorFlow/Keras imports
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tf_keras as keras  # Use tf_keras for TFMOT compatibility
from tf_keras import layers
from tf_keras.preprocessing.image import ImageDataGenerator
import tensorflow_model_optimization as tfmot
import tensorflow.lite as tflite

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print("CPUs:", tf.config.list_physical_devices("CPU"))



TF version: 2.16.1
GPUs: []
CPUs: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [47]:
DATASET_PATH = r"D:\waste_project\waste-classification-system-II\dataset\Garbage Classification Dataset\Garbage classification\Garbage classification"
IMG_SIZE = (96, 96)
NUM_CLASSES = 6

base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
    pooling="avg"
)

base_model.trainable = False  # first train only the head

inputs = keras.Input(shape=(*IMG_SIZE, 3))
x = keras.applications.efficientnet.preprocess_input(inputs)

x = base_model(x, training=False)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()


16705208/16705208 [==============================] - 2s 0us/step
Model: "model_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_12 (InputLayer)       [(None, 96, 96, 3)]       0         
                                                                 
 efficientnetb0 (Functional  (None, 1280)              4049571   
 )                                                               
                                                                 
 dropout_15 (Dropout)        (None, 1280)              0         
                                                                 
 dense_34 (Dense)            (None, 128)               163968    
                                                                 
 dropout_16 (Dropout)        (None, 128)               0         
                                                                 
 dense_35 (Dense)            (None, 6)                 774 

In [31]:
train_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=None,
    color_mode="rgb",
)

val_ds_raw = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=None,
    color_mode="rgb",
)

class_names = train_ds_raw.class_names
num_classes = len(class_names)
print("Classes:", class_names)
print("Num classes:", num_classes)

Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
Num classes: 6


In [41]:
def se_block(x, ratio=8):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, filters))(se)

    se = layers.Dense(
        max(filters // ratio, 4),
        activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(se)

    se = layers.Dense(
        filters,
        activation="sigmoid",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(se)

    return layers.Multiply()([x, se])


In [42]:
def mbconv(x, filters, stride=1, expand=1, drop_rate=0.1):
    reg = keras.regularizers.l2(1e-4)
    in_filters = x.shape[-1]
    shortcut = x

    # Expansion
    if expand > 1:
        x = layers.Conv2D(
            in_filters * expand, 1, padding="same",
            activation="relu6", kernel_regularizer=reg
        )(x)

    # Depthwise conv
    x = layers.DepthwiseConv2D(
        3, strides=stride, padding="same",
        activation="relu6", depthwise_regularizer=reg
    )(x)

    # SE attention
    x = se_block(x)

    # Projection
    x = layers.Conv2D(
        filters, 1, padding="same",
        activation=None, kernel_regularizer=reg
    )(x)

    # Residual
    if stride == 1 and in_filters == filters:
        x = layers.Dropout(drop_rate, noise_shape=(None, 1, 1, 1))(x)
        x = layers.Add()([shortcut, x])

    return x


In [43]:
def build_model(num_classes, img_size=(96, 96)):
    inputs = keras.Input(shape=(*img_size, 3))

    x = layers.Rescaling(1. / 255)(inputs)

    # Stem
    x = layers.Conv2D(
        16, 3, strides=2, padding="same",
        activation="relu6",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)   # 48×48

    # MBConv stages
    x = mbconv(x, 16,  stride=1, expand=1,  drop_rate=0.05)
    x = mbconv(x, 24,  stride=2, expand=6,  drop_rate=0.05)   # 24×24
    x = mbconv(x, 24,  stride=1, expand=6,  drop_rate=0.05)

    x = mbconv(x, 40,  stride=2, expand=6,  drop_rate=0.10)   # 12×12
    x = mbconv(x, 40,  stride=1, expand=6,  drop_rate=0.10)

    x = mbconv(x, 64,  stride=2, expand=6,  drop_rate=0.15)   # 6×6
    x = mbconv(x, 64,  stride=1, expand=6,  drop_rate=0.15)

    # Head
    x = layers.Conv2D(
        128, 1, activation="relu6",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.45)(x)

    x = layers.Dense(
        64, activation="relu",
        kernel_regularizer=keras.regularizers.l2(1e-4)
    )(x)

    x = layers.Dropout(0.30)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)


In [44]:
model = build_model(num_classes=6, img_size=(96, 96))
model.summary()


Model: "model_9"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_10 (InputLayer)       [(None, 96, 96, 3)]          0         []                            
                                                                                                  
 rescaling_9 (Rescaling)     (None, 96, 96, 3)            0         ['input_10[0][0]']            
                                                                                                  
 conv2d_6 (Conv2D)           (None, 48, 48, 16)           448       ['rescaling_9[0][0]']         
                                                                                                  
 depthwise_conv2d (Depthwis  (None, 48, 48, 16)           160       ['conv2d_6[0][0]']            
 eConv2D)                                                                                   

In [46]:
# Compile the MBConv+SE model
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=10,
            restore_best_weights=True,
            mode="max"
        )
    ]
)


Epoch 1/20
64/64 [==============================] - 32s 292ms/step - loss: 1.8502 - accuracy: 0.2226 - val_loss: 1.7841 - val_accuracy: 0.2436
Epoch 2/20
64/64 [==============================] - 16s 242ms/step - loss: 1.7624 - accuracy: 0.2310 - val_loss: 1.7570 - val_accuracy: 0.2436
Epoch 3/20
64/64 [==============================] - 16s 243ms/step - loss: 1.7479 - accuracy: 0.2235 - val_loss: 1.7507 - val_accuracy: 0.2436
Epoch 4/20
64/64 [==============================] - 17s 246ms/step - loss: 1.7405 - accuracy: 0.2201 - val_loss: 1.7474 - val_accuracy: 0.2436
Epoch 5/20
64/64 [==============================] - 17s 256ms/step - loss: 1.7349 - accuracy: 0.2235 - val_loss: 1.7449 - val_accuracy: 0.2436
Epoch 6/20
64/64 [==============================] - 17s 255ms/step - loss: 1.7331 - accuracy: 0.2255 - val_loss: 1.7434 - val_accuracy: 0.2436
Epoch 7/20
64/64 [==============================] - 16s 242ms/step - loss: 1.7344 - accuracy: 0.2221 - val_loss: 1.7425 - val_accuracy: 0.2436